In [1]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("Lab2-Transactions")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")
print(f"Spark {spark.version} — gotowy")

Spark 4.0.0-preview2 — gotowy


In [16]:
df = spark.read.json("data/transactions_10k.jsonl")

print(f"Liczba rekordów: {df.count()}")
df.printSchema()

Liczba rekordów: 10000
root
 |-- amount: double (nullable = true)
 |-- category: string (nullable = true)
 |-- store: string (nullable = true)
 |-- timestamp: string (nullable = true)
 |-- tx_id: string (nullable = true)
 |-- user_id: string (nullable = true)



In [17]:
df.show(10, truncate=False)

+-------+-----------+--------+-------------------+------+-------+
|amount |category   |store   |timestamp          |tx_id |user_id|
+-------+-----------+--------+-------------------+------+-------+
|1989.92|książki    |Gdańsk  |2026-05-11 08:35:24|TX4860|u09    |
|357.81 |książki    |Wrocław |2026-05-11 09:36:38|TX4131|u02    |
|1924.46|odzież     |Wrocław |2026-05-11 09:39:35|TX3305|u17    |
|2011.29|książki    |Gdańsk  |2026-05-11 10:28:12|TX1786|u18    |
|1056.65|żywność    |Warszawa|2026-05-11 08:21:06|TX4422|u20    |
|3534.48|żywność    |Kraków  |2026-05-11 09:52:36|TX9975|u20    |
|4560.35|książki    |Kraków  |2026-05-11 08:17:30|TX9517|u19    |
|853.18 |elektronika|Gdańsk  |2026-05-11 10:41:28|TX9632|u14    |
|1403.01|książki    |Warszawa|2026-05-11 10:17:40|TX1054|u09    |
|2437.35|książki    |Kraków  |2026-05-11 08:53:06|TX3257|u16    |
+-------+-----------+--------+-------------------+------+-------+
only showing top 10 rows



In [18]:
from pyspark.sql.functions import to_timestamp, col

df = df.withColumn("timestamp", to_timestamp(col("timestamp"), "yyyy-MM-dd HH:mm:ss"))

df.printSchema()  # timestamp powinien być teraz 'timestamp (nullable = true)'

root
 |-- amount: double (nullable = true)
 |-- category: string (nullable = true)
 |-- store: string (nullable = true)
 |-- timestamp: timestamp (nullable = true)
 |-- tx_id: string (nullable = true)
 |-- user_id: string (nullable = true)



In [19]:
from pyspark.sql.functions import count, sum as _sum, avg, round as _round

store_summary = (
    df.groupBy("store")
    .agg(
        count("tx_id").alias("liczba_tx"),
        _round(_sum("amount"), 2).alias("suma_PLN"),
        _round(avg("amount"), 2).alias("srednia_PLN"),
    )
    .orderBy("store")
)
store_summary.show()

+--------+---------+----------+-----------+
|   store|liczba_tx|  suma_PLN|srednia_PLN|
+--------+---------+----------+-----------+
|  Gdańsk|     2394|6157463.09|    2572.04|
|  Kraków|     2446|6034215.38|    2466.97|
|Warszawa|     2622|6565341.71|    2503.94|
| Wrocław|     2538|6269062.87|    2470.08|
+--------+---------+----------+-----------+



In [20]:
from pyspark.sql.functions import min as _min, max as _max

store_summary = (
    df.groupBy("category")
    .agg(
        _round(_sum("amount"), 2).alias("suma_PLN"),
        _round(_min("amount"), 2).alias("minimum_PLN"),
        _round(_max("amount"), 2).alias("maksimum_PLN"),
    )
    .orderBy("category")
)
store_summary.show()

+-----------+----------+-----------+------------+
|   category|  suma_PLN|minimum_PLN|maksimum_PLN|
+-----------+----------+-----------+------------+
|elektronika|6399613.26|       5.73|     4995.09|
|    książki|6144892.19|       5.17|     4996.12|
|     odzież|6115737.57|       6.22|     4998.88|
|    żywność|6365840.03|      11.31|     4999.93|
+-----------+----------+-----------+------------+



In [21]:
from pyspark.sql.functions import window

hourly = (
    df.groupBy(window("timestamp", "1 hour"))    # okno 1-godzinne
    .agg(
        count("tx_id").alias("liczba_tx"),
        _round(_sum("amount"), 2).alias("suma_PLN"),
    )
    .orderBy("window")
)
hourly.show(truncate=False)

+------------------------------------------+---------+----------+
|window                                    |liczba_tx|suma_PLN  |
+------------------------------------------+---------+----------+
|{2026-05-11 08:00:00, 2026-05-11 09:00:00}|3347     |8279970.59|
|{2026-05-11 09:00:00, 2026-05-11 10:00:00}|3350     |8405856.86|
|{2026-05-11 10:00:00, 2026-05-11 11:00:00}|3303     |8340255.6 |
+------------------------------------------+---------+----------+



In [22]:
(
    hourly
    .select(
        col("window.start").alias("od"),
        col("window.end").alias("do"),
        "liczba_tx",
        "suma_PLN",
    )
    .show(truncate=False)
)

+-------------------+-------------------+---------+----------+
|od                 |do                 |liczba_tx|suma_PLN  |
+-------------------+-------------------+---------+----------+
|2026-05-11 08:00:00|2026-05-11 09:00:00|3347     |8279970.59|
|2026-05-11 09:00:00|2026-05-11 10:00:00|3350     |8405856.86|
|2026-05-11 10:00:00|2026-05-11 11:00:00|3303     |8340255.6 |
+-------------------+-------------------+---------+----------+



In [26]:
from pyspark.sql.functions import window

half_hourly = (
    df.groupBy(window("timestamp", "30 minutes"), "store")
    .agg(
        count("tx_id").alias("liczba_tx"),
        _round(_sum("amount"), 2).alias("suma_PLN"),
    )
    .orderBy("window", "store")
)
half_hourly.show(truncate=False)

+------------------------------------------+--------+---------+----------+
|window                                    |store   |liczba_tx|suma_PLN  |
+------------------------------------------+--------+---------+----------+
|{2026-05-11 08:00:00, 2026-05-11 08:30:00}|Gdańsk  |387      |1041371.63|
|{2026-05-11 08:00:00, 2026-05-11 08:30:00}|Kraków  |428      |1029527.66|
|{2026-05-11 08:00:00, 2026-05-11 08:30:00}|Warszawa|437      |1107469.06|
|{2026-05-11 08:00:00, 2026-05-11 08:30:00}|Wrocław |429      |1058567.73|
|{2026-05-11 08:30:00, 2026-05-11 09:00:00}|Gdańsk  |385      |923169.78 |
|{2026-05-11 08:30:00, 2026-05-11 09:00:00}|Kraków  |397      |954322.7  |
|{2026-05-11 08:30:00, 2026-05-11 09:00:00}|Warszawa|450      |1135270.25|
|{2026-05-11 08:30:00, 2026-05-11 09:00:00}|Wrocław |434      |1030271.78|
|{2026-05-11 09:00:00, 2026-05-11 09:30:00}|Gdańsk  |438      |1114591.01|
|{2026-05-11 09:00:00, 2026-05-11 09:30:00}|Kraków  |424      |1072828.88|
|{2026-05-11 09:00:00, 20

In [27]:
from pyspark.sql.functions import desc

krakow_top_hour = (
    df.filter(col("store") == "Kraków")
    .groupBy(window("timestamp", "1 hour"))
    .agg(
        _round(_sum("amount"), 2).alias("suma_PLN")
    )
    .orderBy(desc("suma_PLN"))
)
krakow_top_hour.show(1, truncate=False)

+------------------------------------------+----------+
|window                                    |suma_PLN  |
+------------------------------------------+----------+
|{2026-05-11 09:00:00, 2026-05-11 10:00:00}|2094912.69|
+------------------------------------------+----------+
only showing top 1 row



In [28]:
sliding = (
    df.groupBy(window("timestamp", "1 hour", "30 minutes"))  # szerokość 1h, krok 30min
    .agg(
        count("tx_id").alias("liczba_tx"),
        _round(_sum("amount"), 2).alias("suma_PLN"),
    )
    .select(
        col("window.start").alias("od"),
        col("window.end").alias("do"),
        "liczba_tx",
        "suma_PLN",
    )
    .orderBy("od")
)
sliding.show(truncate=False)

+-------------------+-------------------+---------+----------+
|od                 |do                 |liczba_tx|suma_PLN  |
+-------------------+-------------------+---------+----------+
|2026-05-11 07:30:00|2026-05-11 08:30:00|1681     |4236936.08|
|2026-05-11 08:00:00|2026-05-11 09:00:00|3347     |8279970.59|
|2026-05-11 08:30:00|2026-05-11 09:30:00|3411     |8396365.05|
|2026-05-11 09:00:00|2026-05-11 10:00:00|3350     |8405856.86|
|2026-05-11 09:30:00|2026-05-11 10:30:00|3284     |8293119.47|
|2026-05-11 10:00:00|2026-05-11 11:00:00|3303     |8340255.6 |
|2026-05-11 10:30:00|2026-05-11 11:30:00|1624     |4099662.45|
+-------------------+-------------------+---------+----------+



In [30]:
tumbling_rows = (
    df.groupBy(window("timestamp", "1 hour"))
    .agg(count("tx_id"))
    .count()
)
sliding_rows = (
    df.groupBy(window("timestamp", "1 hour", "30 minutes"))
    .agg(count("tx_id"))
    .count()
)
print(f"Tumbling (1h):          {tumbling_rows} okien")
print(f"Sliding  (1h / 30min):  {sliding_rows} okien")

# Odpowiedz w komentarzu: dlaczego sliding ma więcej wierszy?
# TWOJA ODPOWIEDŹ:
# Ponieważ okna się na siebie nakładają

Tumbling (1h):          3 okien
Sliding  (1h / 30min):  7 okien


In [31]:
# Odpowiedz na pytania w komentarzach:

# 1. Ile transakcji jest w oknie 09:00–10:00?
#    Sprawdź w wyniku zadania 3.1.
#    ODPOWIEDŹ: 3350

# 2. Jaka jest różnica między groupBy("store") a groupBy(window(...), "store")?
#    ODPOWIEDŹ: groupBy store grupuje po sklepie a groupBy window, store po sklepie w oknie czasowym

# 3. W oknie sliding 1h/30min — ile okien zawiera transakcje z godziny 09:30?
#    Wskazówka: narysuj oś czasu.
#    ODPOWIEDŹ: 2 okna

In [34]:
#ZAD dom 1
from pyspark.sql.functions import desc

gdansk_worst_mean_amount = (
    df.filter(col("store") == "Gdańsk")
    .groupBy(window("timestamp", "1 hour"))
    .agg(
        _round(avg("amount"), 2).alias("średnia_kwota_PLN")
    )
    .orderBy("średnia_kwota_PLN")
)
gdansk_worst_mean_amount.show(1, truncate=False)

+------------------------------------------+-----------------+
|window                                    |średnia_kwota_PLN|
+------------------------------------------+-----------------+
|{2026-05-11 08:00:00, 2026-05-11 09:00:00}|2544.74          |
+------------------------------------------+-----------------+
only showing top 1 row



In [36]:
from pyspark.sql.functions import col, hour, minute, count, desc

category_9_930 = (
    df.filter(
        (hour(col("timestamp")) == 9) & 
        (minute(col("timestamp")) < 30)
    )
    .groupBy("category")
    .agg(
        count("tx_id").alias("liczba_transakcji"),
    )
    .orderBy(desc("liczba_transakcji"))
)
category_9_930.show(truncate=False)

+-----------+-----------------+
|category   |liczba_transakcji|
+-----------+-----------------+
|elektronika|471              |
|książki    |434              |
|odzież     |424              |
|żywność    |416              |
+-----------+-----------------+



In [37]:
szczyt_15_min = (
    df.groupBy(window("timestamp", "15 minutes"))
    .agg(
        count("tx_id").alias("liczba_transakcji")
    )
    .orderBy(desc("liczba_transakcji"))
)
szczyt_15_min.show(1, truncate=False)

+------------------------------------------+-----------------+
|window                                    |liczba_transakcji|
+------------------------------------------+-----------------+
|{2026-05-11 09:15:00, 2026-05-11 09:30:00}|886              |
+------------------------------------------+-----------------+
only showing top 1 row

